# Предпочтения и дистилляция

Разница между парными предпочтениями, наградой и ответами учителя.

> Все примеры этого урока работают на CPU, не скачивают модели и используют искусственные данные. Они объясняют механизм; качество настоящего обучения проверяется отдельно.

## Что важно понять

SFT учится по правильным ответам. DPO использует пары chosen/rejected и отношение policy к фиксированной reference. Distillation переносит распределение учителя через KL. Это разные источники сигнала, а не взаимозаменяемые названия.

В примере считаем DPO на sequence log-probabilities и distillation на logits. Для текста нужны суммы log-probs только по нужным токенам, одинаковый tokenizer для прямого logit distillation и независимая оценка качества. Доступность конкретного режима в студии зависит от выбранного семейства модели.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Запустите эксперимент

Запускайте ячейки по порядку. Код можно менять; Shift + Enter выполняет текущую ячейку.

In [ ]:
chosen = torch.tensor([-2.0, -1.4], requires_grad=True)
rejected = torch.tensor([-3.0, -2.0], requires_grad=True)
ref_chosen = torch.tensor([-2.5, -1.7]); ref_rejected = torch.tensor([-2.8, -2.1])
beta = 0.1
dpo_loss = -F.logsigmoid(beta * ((chosen - rejected) - (ref_chosen - ref_rejected))).mean()
dpo_loss.backward()
print('DPO:', dpo_loss.item(), 'chosen gradient:', chosen.grad)
teacher = torch.tensor([[3., 1., -1.]])
student = torch.tensor([[0., 0., 0.]], requires_grad=True)
temperature = 2.
kd = F.kl_div((student / temperature).log_softmax(-1), (teacher / temperature).softmax(-1), reduction='batchmean') * temperature**2
kd.backward(); print('distillation KL:', kd.item(), 'student gradient:', student.grad)

## Попробуйте сами

Поменяйте chosen и rejected местами. Проверьте знак градиента. Затем меняйте temperature: какие различия в распределении учителя становятся заметнее?

Сохраните копию ноутбука и запишите, что изменили, чего ожидали и что измерили.

## Продолжить в студии

Кнопка «Открыть раздел студии» над уроком ведёт к связанным инструментам. Учебный код сам не запускает большие тренировки. Перед реальным запуском проверьте данные, ресурсы и поддержку метода у выбранной модели.

## Источники

- [DPO](https://arxiv.org/abs/2305.18290)
- [Distillation](https://arxiv.org/abs/1503.02531)